<a href="https://colab.research.google.com/github/ROHAN-BHUTANI/MediTriageAI/blob/main/EPATH_CO_REASON_Training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# E-PATH-CO-REASON: Production-Grade Google Colab Training Notebook

This notebook serves as the **unified, production-grade training and evaluation pipeline** for the E-PATH-CO-REASON research experiments. It is designed to run end-to-end on a completely fresh Google Colab GPU runtime (or a local environment) with a single "Run All" command.

## 1. Notebook Purpose
This notebook orchestrates the training, validation, evaluation, and logging of the **E-PATH-CO-REASON** model. It includes mechanisms for differentiable path routing via Gumbel-Softmax, representation alignment via Dynamic Consistency Projection (DCP), and multi-objective composite loss tracking.

## 2. Directory Layout & Persistence
All outputs are persistently saved inside the experiments workspace folder:
```
experiments/<experiment_name>/
├── best_model.pt             # Best model checkpoint
├── latest_model.pt           # Last completed epoch checkpoint
├── checkpoints/              # Checkpoint directory
├── logs/                     # environment.json and configuration.json
├── exports/                  # training_history.csv, metrics.json, routing_statistics.json
└── figures/                  # loss_curves.png, accuracy_curves.png, confusion matrices
```

## 3. Running & Resuming
- **First Run**: Select `Runtime -> Run All`. The notebook will automatically check imports, setup the repository, validate the dataset, and start training from scratch.
- **Resuming Interrupted Runs**: If training gets disconnected, select `Runtime -> Run All` again. The notebook automatically mounts Google Drive, detects the existing `latest_model.pt` checkpoint, and resumes training from the exact interrupted epoch/optimizer/scheduler/seed state.

## 1. Central Experiment Configuration Block

In [36]:
# ==========================================
# CENTRAL EXPERIMENT CONFIGURATION BLOCK
# ==========================================
EXPERIMENT_CONFIG = {
    "experiment_name": "epath_co_reason_baseline",
    "git_branch": "main",  # Branch to clone/checkout if not already local

    # Dataset Parameters
    "dataset_relative_path": "meditriage/data/processed/dataset.csv",
    "max_samples": None,  # Set to an integer (e.g. 500) to train on subset, or None for the full dataset
    "max_length": 128,

    # Split Parameters
    "train_ratio": 0.8,
    "val_ratio": 0.1,
    "test_ratio": 0.1,

    # Trainer Parameters
    "epochs": 10,
    "batch_size": 32,
    "learning_rate": 1e-4,
    "encoder_lr": 2e-5,
    "weight_decay": 0.01,
    "gradient_clipping": 1.0,
    "gradient_accumulation_steps": 1,
    "use_amp": True,
    "seed": 1337,
    "optimizer_type": "adamw",
    "scheduler_type": "cosine",
    "warmup_ratio": 0.1,

    # Early Stopping
    "early_stopping_patience": 3,
    "early_stopping_metric": "val_loss",
    "early_stopping_min_improvement": 1e-4,

    # Storage and Environment
    "use_drive": True,
    "drive_workspace_dir": "/content/drive/MyDrive/MediTriageAI"
}

In [37]:
!find /content/MediTriageAI/models -name "__init__.py"

/content/MediTriageAI/models/__init__.py
/content/MediTriageAI/models/emergent_path_triage/__init__.py


In [38]:
!find /content/MediTriageAI/models -name "__init__.py"

/content/MediTriageAI/models/__init__.py
/content/MediTriageAI/models/emergent_path_triage/__init__.py


In [39]:
from pathlib import Path
import os
import sys

REPO_ROOT = Path("/content/MediTriageAI")

os.chdir(REPO_ROOT)

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

print(f"Repository Root : {REPO_ROOT}")
print(f"Working Directory : {os.getcwd()}")

Repository Root : /content/MediTriageAI
Working Directory : /content/MediTriageAI


In [40]:
!pwd
!ls -la

/content/MediTriageAI
total 328
drwxr-xr-x 12 root root  4096 Jul 21 09:36 .
drwxr-xr-x  1 root root  4096 Jul 21 09:36 ..
drwxr-xr-x  3 root root  4096 Jul 21 09:36 analysis
-rw-r--r--  1 root root 26555 Jul 21 09:36 analysis_report.html
-rw-r--r--  1 root root 18412 Jul 21 09:36 analysis_report.md
-rw-r--r--  1 root root  4629 Jul 21 09:36 ANNOTATION_INTEGRITY_CHECK.md
-rw-r--r--  1 root root  8117 Jul 21 09:36 BASELINE_RESULTS.md
-rw-r--r--  1 root root  2461 Jul 21 09:36 BUGFIX_VERIFICATION.md
-rw-r--r--  1 root root   319 Jul 21 09:36 CITATION.cff
-rw-r--r--  1 root root  1685 Jul 21 09:36 CLASS_DISTRIBUTION.md
-rw-r--r--  1 root root    38 Jul 21 09:36 clinician_overlap_stats.json
-rw-r--r--  1 root root   151 Jul 21 09:36 CLINICIAN_TEST_SET_V3.md
-rw-r--r--  1 root root  2219 Jul 21 09:36 COLAB_SETUP.md
-rw-r--r--  1 root root  5778 Jul 21 09:36 CONFUSION_ANALYSIS.md
drwxr-xr-x  5 root root  4096 Jul 21 09:36 dashboard_web
-rw-r--r--  1 root root  6760 Jul 21 09:36 DEMO_SCRIPT.m

In [41]:
!find /content/MediTriageAI -maxdepth 3 -type d

/content/MediTriageAI
/content/MediTriageAI/meditriage
/content/MediTriageAI/meditriage/data
/content/MediTriageAI/meditriage/data/processed
/content/MediTriageAI/meditriage/data/raw
/content/MediTriageAI/.git
/content/MediTriageAI/.git/info
/content/MediTriageAI/.git/refs
/content/MediTriageAI/.git/refs/heads
/content/MediTriageAI/.git/refs/tags
/content/MediTriageAI/.git/refs/remotes
/content/MediTriageAI/.git/logs
/content/MediTriageAI/.git/logs/refs
/content/MediTriageAI/.git/hooks
/content/MediTriageAI/.git/objects
/content/MediTriageAI/.git/objects/info
/content/MediTriageAI/.git/objects/pack
/content/MediTriageAI/.git/branches
/content/MediTriageAI/dashboard_web
/content/MediTriageAI/dashboard_web/data
/content/MediTriageAI/dashboard_web/css
/content/MediTriageAI/dashboard_web/js
/content/MediTriageAI/src
/content/MediTriageAI/analysis
/content/MediTriageAI/analysis/results
/content/MediTriageAI/analysis/results/experiment_2026_07_20_070026
/content/MediTriageAI/analysis/results

## 2. Automatic Repository Setup & Import Validation

In [42]:
# ==========================================
# SECTION 1 & 3: REPOSITORY DETECT & IMPORT CHECKS
# ==========================================
import os
import sys
from pathlib import Path

# Walk up parent tree to find repo root
def find_repo_root(start_dir: Path) -> Path:
    for parent in [start_dir] + list(start_dir.parents):
        if (parent / ".git").exists() or (parent / "requirements.txt").exists():
            return parent
    return start_dir

curr_dir = Path(os.getcwd()).resolve()
repo_root = find_repo_root(curr_dir)
print(f"Repository Root Detected: {repo_root}")

if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

# Clone check (for fresh Colab runtimes)
if not (repo_root / "requirements.txt").exists():
    REPO_URL = "https://github.com/ROHAN-BHUTANI/MediTriageAI.git"
    REPO_DIR = "MediTriageAI_Data_Engine"
    print(f"Cloning fresh repository: {REPO_URL}...")
    !git clone -b {EXPERIMENT_CONFIG['git_branch']} {REPO_URL}
    repo_root = Path(os.getcwd()).resolve() / REPO_DIR
    if str(repo_root) not in sys.path:
        sys.path.insert(0, str(repo_root))

# Install dependencies
req_path = repo_root / "requirements.txt"
if req_path.exists():
    print(f"Installing dependencies from: {req_path}")
    !pip install -q -r {req_path}
else:
    print("Warning: requirements.txt not found. Performing fallback install...")
    !pip install -q transformers scikit-learn matplotlib seaborn pandas numpy torch psutil

# Verify Imports
print("Validating imports and workspace modules...")
required_imports = [
    ("torch", "torch"),
    ("transformers", "transformers"),
    ("pandas", "pandas"),
    ("numpy", "numpy"),
    ("sklearn", "scikit-learn"),
    ("matplotlib", "matplotlib"),
    ("seaborn", "seaborn"),
    ("psutil", "psutil"),
    ("models.emergent_path_triage.model", "E-PATH-CO-REASON Model"),
    ("src.data_pipeline", "E-PATH-CO-REASON Data Pipeline"),
    ("src.trainer", "E-PATH-CO-REASON Trainer")
]

missing = []
for mod_name, friendly_name in required_imports:
    try:
        __import__(mod_name)
        print(f"  ✓ {mod_name} imported successfully.")
    except ImportError as e:
        print(f"  ✗ Failed to import {mod_name}: {e}")
        missing.append(friendly_name)

if missing:
    raise ImportError(f"Verification failed. Missing required components: {missing}")
print("All imports validated successfully.")

Repository Root Detected: /content/MediTriageAI
Installing dependencies from: /content/MediTriageAI/requirements.txt
Validating imports and workspace modules...
  ✓ torch imported successfully.
  ✓ transformers imported successfully.
  ✓ pandas imported successfully.
  ✓ numpy imported successfully.
  ✓ sklearn imported successfully.
  ✓ matplotlib imported successfully.
  ✓ seaborn imported successfully.
  ✓ psutil imported successfully.
  ✓ models.emergent_path_triage.model imported successfully.
  ✓ src.data_pipeline imported successfully.
  ✓ src.trainer imported successfully.
All imports validated successfully.


## 3. Environment Validation

In [43]:
# ==========================================
# SECTION 2: ENVIRONMENT VALIDATION
# ==========================================
import json
import torch
import psutil
import transformers
from src.data_pipeline import detect_colab_environment

env_meta = detect_colab_environment()
has_gpu = env_meta["has_gpu"]
gpu_name = env_meta["gpu_name"]
total_vram = 0
free_vram = 0

if has_gpu:
    t = torch.cuda.get_device_properties(0).total_memory
    a = torch.cuda.memory_allocated(0)
    total_vram = t / (1024 ** 3)
    free_vram = (t - a) / (1024 ** 3)

git_commit = "N/A"
try:
    import subprocess
    git_commit = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=str(repo_root)).decode("utf-8").strip()
except Exception:
    pass

env_info = {
    "python_version": sys.version,
    "pytorch_version": torch.__version__,
    "cuda_version": torch.version.cuda if has_gpu else "N/A",
    "transformers_version": transformers.__version__,
    "gpu_model": gpu_name,
    "total_gpu_memory_gb": total_vram,
    "available_gpu_memory_gb": free_vram,
    "cpu_cores": psutil.cpu_count(logical=True),
    "ram_gb": psutil.virtual_memory().total / (1024 ** 3),
    "git_commit_hash": git_commit
}

print("ENVIRONMENT AUDIT LOG:")
print(json.dumps(env_info, indent=4))

ENVIRONMENT AUDIT LOG:
{
    "python_version": "3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]",
    "pytorch_version": "2.11.0+cpu",
    "cuda_version": "N/A",
    "transformers_version": "5.13.1",
    "gpu_model": "N/A",
    "total_gpu_memory_gb": 0,
    "available_gpu_memory_gb": 0,
    "cpu_cores": 2,
    "ram_gb": 12.671417236328125,
    "git_commit_hash": "d91639e6f8d1ea866a52d81196403ca1a37c1023"
}


## 4. Google Drive Mount & Workspace Setup

In [44]:
# ==========================================
# SECTION 3: DRIVE MOUNT & FOLDERS VERIFICATION
# ==========================================
from pathlib import Path

try:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_AVAILABLE = True
except Exception as e:
    print(f"Drive mount failed: {e}")
    print("Using local /content workspace instead.")
    DRIVE_AVAILABLE = False

# -------------------------
# Workspace selection
# -------------------------
if DRIVE_AVAILABLE:
    drive_base = Path(EXPERIMENT_CONFIG["drive_workspace_dir"])
else:
    drive_base = Path("/content/MediTriageAI")

print(f"Workspace: {drive_base}")

exp_base = drive_base / "experiments" / EXPERIMENT_CONFIG["experiment_name"]

dirs = {
    "experiments": exp_base,
    "checkpoints": exp_base / "checkpoints",
    "logs": exp_base / "logs",
    "figures": exp_base / "figures",
    "exports": exp_base / "exports"
}

for name, path in dirs.items():
    path.mkdir(parents=True, exist_ok=True)
    print(f"Verified folder '{name}': {path}")

# Write permission check
test_file = exp_base / "write_check.txt"
try:
    test_file.write_text("write check successful")
    test_file.unlink()
    print("Write permissions successfully verified.")
except Exception as e:
    raise PermissionError(f"Target folder is not writable: {e}")

# Save environment.json
with open(dirs["logs"] / "environment.json", "w", encoding="utf-8") as f:
    json.dump(env_info, f, indent=4)

Mounted at /content/drive
Workspace: /content/drive/MyDrive/MediTriageAI
Verified folder 'experiments': /content/drive/MyDrive/MediTriageAI/experiments/epath_co_reason_baseline
Verified folder 'checkpoints': /content/drive/MyDrive/MediTriageAI/experiments/epath_co_reason_baseline/checkpoints
Verified folder 'logs': /content/drive/MyDrive/MediTriageAI/experiments/epath_co_reason_baseline/logs
Verified folder 'figures': /content/drive/MyDrive/MediTriageAI/experiments/epath_co_reason_baseline/figures
Verified folder 'exports': /content/drive/MyDrive/MediTriageAI/experiments/epath_co_reason_baseline/exports
Write permissions successfully verified.


## 5. Dataset Validation Checks

In [45]:
# ==========================================
# SECTION 4: DATASET VALIDATION
# ==========================================
import pandas as pd
from src.data_pipeline import LabelValidator
from transformers import AutoTokenizer

print("Discovering dataset relative to repo root...")
dataset_csv = repo_root / EXPERIMENT_CONFIG["dataset_relative_path"]

if not dataset_csv.exists():
    raise FileNotFoundError(f"Dataset CSV not found at: {dataset_csv}")

df = pd.read_csv(dataset_csv)
print(f"Loaded dataset with {len(df)} total rows.")

# Column validations
required_cols = ["text", "department_code", "severity_heuristic"]
for col in required_cols:
    if col not in df.columns:
        raise KeyError(f"Dataset schema validation failed. Column '{col}' is missing.")

# Drop NaNs in text
nan_text = df["text"].isna().sum()
if nan_text > 0:
    print(f"Removing {nan_text} rows with missing text...")
    df = df.dropna(subset=["text"])

# Mappings validations
validator = LabelValidator()
invalid_spec = (~df["department_code"].isin(validator.specialist_classes)).sum()
invalid_sev = (~df["severity_heuristic"].isin(validator.severity_labels)).sum()

if invalid_spec > 0:
    raise ValueError(f"Invalid specialty labels count: {invalid_spec}")
if invalid_sev > 0:
    raise ValueError(f"Invalid severity labels count: {invalid_sev}")

print("Validating tokenizer encoding format...")
tokenizer = AutoTokenizer.from_pretrained("xlm-roberta-base")
try:
    tokens = tokenizer.encode(df["text"].iloc[0], truncation=True, max_length=EXPERIMENT_CONFIG["max_length"])
    print(f"Tokenizer validation passed. First text encode length: {len(tokens)}")
except Exception as e:
    raise RuntimeError(f"Tokenizer compatibility check failed: {e}")

print("Dataset validation validation PASSED.")

Discovering dataset relative to repo root...
Loaded dataset with 19996 total rows.
Removing 33 rows with missing text...
Validating tokenizer encoding format...


config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.10M [00:00<?, ?B/s]

Tokenizer validation passed. First text encode length: 128
Dataset validation validation PASSED.


## 6. Training Initialization & Checkpoint Resume Loop

In [46]:
!grep -n "def load_checkpoint" /content/MediTriageAI/src/trainer.py

401:    def load_checkpoint(self, path: Path) -> int:


In [47]:
!sed -n '250,420p' /content/MediTriageAI/src/trainer.py

                self.optimizer.zero_grad()
                if self.scheduler is not None:
                    self.scheduler.step()

            # Record metrics
            spec_preds = outputs.specialist_logits.argmax(dim=-1)
            sev_preds = outputs.severity_logits.argmax(dim=-1)
            tracker.update(loss_dict, spec_preds, labels_spec, sev_preds, labels_sev)

        metrics = tracker.get_summary()
        metrics["lr"] = self.optimizer.param_groups[-1]["lr"]
        return metrics

    def validate(self) -> dict[str, float]:
        """Perform evaluation pass over validation split."""
        self.model.eval()
        tracker = MetricTracker()

        with torch.no_grad():
            for batch in self.val_loader:
                input_ids = batch["input_ids"].to(self.device)
                attention_mask = batch["attention_mask"].to(self.device)
                labels_spec = batch["labels_specialist"].to(self.device)
                labels_sev = batch["labels_severi

In [48]:
from pathlib import Path

print(dirs["checkpoints"])
print((dirs["checkpoints"] / "best_model.pt").exists())
print((dirs["checkpoints"] / "latest_model.pt").exists())

/content/drive/MyDrive/MediTriageAI/experiments/epath_co_reason_baseline/checkpoints
True
True


In [52]:
!grep -Rn "trainer =" /content/MediTriageAI

/content/MediTriageAI/EPATH_CO_REASON_Training.ipynb:427:    "trainer = EmergentTrainer(\n",
/content/MediTriageAI/scripts/run_baseline.py:120:    trainer = EmergentTrainer(
/content/MediTriageAI/tests/test_emergent_path_triage.py:1612:    trainer = EmergentTrainer(
/content/MediTriageAI/tests/test_emergent_path_triage.py:1631:    new_trainer = EmergentTrainer(
/content/MediTriageAI/tests/test_emergent_path_triage.py:1686:    trainer = EmergentTrainer(


In [53]:
!grep -Rn "test_loader" /content/MediTriageAI

/content/MediTriageAI/EPATH_CO_REASON_Training.ipynb:401:    "test_loader = get_dataloader(create_ds(test_df), batch_size=EXPERIMENT_CONFIG[\"batch_size\"], shuffle=False)\n",
/content/MediTriageAI/EPATH_CO_REASON_Training.ipynb:483:    "    for batch in test_loader:\n",
grep: /content/MediTriageAI/src/__pycache__/trainer.cpython-312.pyc: binary file matches
/content/MediTriageAI/src/trainer.py:124:        test_loader: DataLoader | None = None,
/content/MediTriageAI/src/trainer.py:131:        self.test_loader = test_loader
/content/MediTriageAI/scripts/colab_train.py:62:    test_loader = DataLoader(test_dataset, batch_size=32)
/content/MediTriageAI/scripts/colab_train.py:147:        for batch in test_loader:
/content/MediTriageAI/scripts/run_experiment.py:193:    metrics = evaluator.run_evaluation(artifacts.model, artifacts.tokenizer, artifacts.test_loader, artifacts.config)
/content/MediTriageAI/scripts/train.py:69:    test_loader: DataLoader
/content/MediTriageAI/scripts/train.py:96:

In [54]:
!grep -Rn "model =" /content/MediTriageAI

/content/MediTriageAI/EPATH_CO_REASON_Training.ipynb:406:    "model = model_meta.build(None, triage_config=config)\n",
/content/MediTriageAI/src/trainer.py:127:        self.model = model
/content/MediTriageAI/src/metrics.py:195:    novel_model = next((item for item in ranked if item.get("is_novel_contribution")), ranked[0])
/content/MediTriageAI/analysis/io.py:70:    built_model = model_instance.build(None)
/content/MediTriageAI/scripts/colab_train.py:66:    model = DistilBertClassifier(len(label_list)).to(device)
/content/MediTriageAI/scripts/train.py:88:    built_model = model_meta.build(None)
/content/MediTriageAI/scripts/serve_dashboard.py:108:            model = data.get("model", "xlm_roberta")
/content/MediTriageAI/scripts/diagnose_baseline.py:86:    model = model_meta.build(TinyConfig(), triage_config=config)
/content/MediTriageAI/scripts/run_baseline.py:112:    model = model_meta.build(TinyConfig(), triage_config=config)
/content/MediTriageAI/scripts/infer.py:89:    model = MOD

In [50]:
for var in [
    "trainer",
    "model",
    "test_loader",
    "validator",
    "dirs",
    "exp_base",
]:
    print(f"{var}: {'YES' if var in globals() else 'NO'}")

trainer: NO
model: NO
test_loader: NO
validator: YES
dirs: YES
exp_base: YES


## 8. Post-Training Evaluation Exports

In [49]:
# ==========================================
# SECTION 8: EVALUATION METRICS & CONFUSION PLOTS
# ==========================================
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, precision_recall_fscore_support, accuracy_score

print("Loading best parameters checkpoint...")
best_ckpt = dirs["checkpoints"] / "best_model.pt"
trainer.load_checkpoint(best_ckpt)

model.eval()
all_spec_labels = []
all_sev_labels = []
all_spec_preds = []
all_sev_preds = []
all_routing_probs = []

with torch.no_grad():
    for batch in test_loader:
        input_ids = batch["input_ids"].to(trainer.device)
        attention_mask = batch["attention_mask"].to(trainer.device)
        labels_spec = batch["labels_specialist"]
        labels_sev = batch["labels_severity"]

        outputs = model(input_ids, attention_mask)

        spec_preds = outputs.specialist_logits.argmax(dim=-1).cpu().numpy()
        sev_preds = outputs.severity_logits.argmax(dim=-1).cpu().numpy()

        all_spec_labels.extend(labels_spec.numpy())
        all_sev_labels.extend(labels_sev.numpy())
        all_spec_preds.extend(spec_preds)
        all_sev_preds.extend(sev_preds)

        if model._last_routing_decision is not None:
            all_routing_probs.append(model._last_routing_decision.routing_probabilities.cpu().numpy())

spec_acc = accuracy_score(all_spec_labels, all_spec_preds)
spec_p, spec_r, spec_f1, _ = precision_recall_fscore_support(all_spec_labels, all_spec_preds, average="macro", zero_division=0)

sev_acc = accuracy_score(all_sev_labels, all_sev_preds)
sev_p, sev_r, sev_f1, _ = precision_recall_fscore_support(all_sev_labels, all_sev_preds, average="macro", zero_division=0)

metrics_export = {
    "specialist": {
        "accuracy": float(spec_acc),
        "macro_precision": float(spec_p),
        "macro_recall": float(spec_r),
        "macro_f1": float(spec_f1)
    },
    "severity": {
        "accuracy": float(sev_acc),
        "macro_precision": float(sev_p),
        "macro_recall": float(sev_r),
        "macro_f1": float(sev_f1)
    },
    "overall_losses": {
        "val_loss": float(best_val_metrics["val_loss"]),
        "val_specialist_loss": float(best_val_metrics["val_specialist_loss"]),
        "val_severity_loss": float(best_val_metrics["val_severity_loss"]),
        "val_cons_loss": float(best_val_metrics["val_cons_loss"]),
        "val_div_loss": float(best_val_metrics["val_div_loss"]),
        "val_ortho_loss": float(best_val_metrics["val_ortho_loss"])
    }
}

with open(dirs["exports"] / "metrics.json", "w", encoding="utf-8") as f:
    json.dump(metrics_export, f, indent=4)

if all_routing_probs:
    all_probs = np.concatenate(all_routing_probs, axis=0)
    B_t, M_t, N_t = all_probs.shape
    epsilon = 1e-9
    entropies = -np.sum(all_probs * np.log(all_probs + epsilon), axis=-1)
    util_argmax = all_probs.argmax(axis=-1)
    utilization_counts = [np.bincount(util_argmax[:, s], minlength=N_t).tolist() for s in range(M_t)]

    routing_export = {
        "mean_routing_entropy": float(entropies.mean()),
        "entropy_per_step": entropies.mean(axis=0).tolist(),
        "ctb_utilizations_per_step": utilization_counts,
        "average_reasoning_depth": M_t,
        "mean_confidence": float(np.max(all_probs, axis=-1).mean())
    }
    with open(dirs["exports"] / "routing_statistics.json", "w", encoding="utf-8") as f:
        json.dump(routing_export, f, indent=4)

history_df = pd.DataFrame(trainer.history)
history_df.to_csv(dirs["exports"] / "training_history.csv", index=False)
val_cols = [c for c in history_df.columns if "val_" in c or c in ["epoch", "time"]]
history_df[val_cols].to_csv(dirs["exports"] / "validation_history.csv", index=False)

plt.figure(figsize=(10, 5))
plt.plot(history_df["epoch"], history_df["train_loss"], label="Train Loss", marker="o")
plt.plot(history_df["epoch"], history_df["val_loss"], label="Val Loss", marker="x")
plt.title("E-PATH-CO-REASON Loss Curves")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.grid(True)
plt.legend()
plt.savefig(dirs["figures"] / "loss_curves.png")
plt.close()

plt.figure(figsize=(10, 5))
plt.plot(history_df["epoch"], history_df["train_specialist_acc"], label="Train Spec Acc", marker="o")
plt.plot(history_df["epoch"], history_df["val_specialist_acc"], label="Val Spec Acc", marker="x")
plt.plot(history_df["epoch"], history_df["train_severity_acc"], label="Train Sev Acc", marker="s")
plt.plot(history_df["epoch"], history_df["val_severity_acc"], label="Val Sev Acc", marker="d")
plt.title("E-PATH-CO-REASON Accuracy Curves")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.grid(True)
plt.legend()
plt.savefig(dirs["figures"] / "accuracy_curves.png")
plt.close()

plt.figure(figsize=(10, 8))
sns.heatmap(confusion_matrix(all_spec_labels, all_spec_preds), annot=True, fmt="d", cmap="Blues",
            xticklabels=validator.specialist_classes, yticklabels=validator.specialist_classes)
plt.title("Specialist Confusion Matrix")
plt.savefig(dirs["figures"] / "specialist_confusion_matrix.png")
plt.close()

plt.figure(figsize=(8, 6))
sns.heatmap(confusion_matrix(all_sev_labels, all_sev_preds), annot=True, fmt="d", cmap="Oranges",
            xticklabels=validator.severity_labels, yticklabels=validator.severity_labels)
plt.title("Severity Confusion Matrix")
plt.savefig(dirs["figures"] / "severity_confusion_matrix.png")
plt.close()

shutil.copyfile(dirs["checkpoints"] / "best_model.pt", exp_base / "best_model.pt")
shutil.copyfile(dirs["checkpoints"] / "latest_model.pt", exp_base / "latest_model.pt")

print("Post-training assets successfully compiled and exported.")

Loading best parameters checkpoint...


NameError: name 'trainer' is not defined

## 9. Final Experiment Summary Report

In [ ]:
# ==========================================
# SECTION 9: FINAL EXPERIMENT SUMMARY
# ==========================================
print(f"==================================================")
print(f"FINAL EXPERIMENT SUMMARY REPORT")
print(f"==================================================")
print(f"Experiment Name      : {EXPERIMENT_CONFIG['experiment_name']}")
print(f"GPU Model Used       : {env_info['gpu_model']}")
print(f"Best Training Epoch  : {best_val_metrics.get('epoch', 'N/A')}")
print(f"Specialist Accuracy  : {spec_acc:.2%}")
print(f"Severity Accuracy    : {sev_acc:.2%}")
print(f"Specialist F1-Score  : {spec_f1:.2%}")
print(f"Severity F1-Score    : {sev_f1:.2%}")
print(f"--------------------------------------------------")
print(f"Outputs Directory Locations:")
print(f"- Checkpoints        : {dirs['checkpoints']}")
print(f"- Exported Reports   : {dirs['exports']}")
print(f"- Plot Figures       : {dirs['figures']}")
print(f"==================================================")